# Feature Engineering

This notebook creates model-ready processed datasets from the cleaned transaction data. Features are calculated before aggregation, then daily and weekly product datasets are produced for chronological train/test splitting. Customer RFM is generated as monthly as-of snapshots so future transactions cannot enter an earlier customer profile.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_PATH = PROJECT_ROOT / "data" / "interim" / "clean_transactions.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ASSUMED_MARGIN = 0.30

transactions = pd.read_csv(INPUT_PATH, parse_dates=["InvoiceDate"])
transactions["Invoice"] = transactions["Invoice"].astype("string")
transactions["StockCode"] = transactions["StockCode"].astype("string")
transactions["Customer ID"] = transactions["Customer ID"].astype("string")
transactions["InvoiceDate"] = pd.to_datetime(transactions["InvoiceDate"], errors="coerce")

transactions["Date"] = transactions["InvoiceDate"].dt.date
transactions["Time"] = transactions["InvoiceDate"].dt.strftime("%H:%M:%S")
transactions["Hour"] = transactions["InvoiceDate"].dt.hour.astype("Int64")
transactions["DayOfWeek"] = transactions["InvoiceDate"].dt.dayofweek.astype("Int64")
transactions["Month"] = transactions["InvoiceDate"].dt.month.astype("Int64")
transactions["Year"] = transactions["InvoiceDate"].dt.year.astype("Int64")
transactions["IsCancelled"] = transactions["Invoice"].str.startswith("C", na=False)
transactions["IsReturn"] = transactions["Quantity"].lt(0)
transactions["TotalPrice"] = transactions["Quantity"] * transactions["Price"]
transactions["EstimatedProfit"] = transactions["TotalPrice"] * ASSUMED_MARGIN

# Positive, non-operational sales are the demand basis for product models.
product_sales = transactions.loc[
    transactions["Quantity"].gt(0)
    & ~transactions["IsCancelled"]
    & ~transactions["is_operational_code"]
].copy()

transactions.to_csv(PROCESSED_DIR / "feature_transactions.csv", index=False)
transactions.shape, product_sales.shape

((1013408, 26), (987719, 26))

In [2]:
def aggregate_products(data, period):
    """Summarize demand at the product and time grain used by forecasting models."""
    grouped = (
        data.assign(Period=data["InvoiceDate"].dt.to_period(period).dt.start_time)
        .groupby(["Period", "StockCode", "Description"], as_index=False)
        .agg(
            Units=("Quantity", "sum"),
            Revenue=("TotalPrice", "sum"),
            EstimatedProfit=("EstimatedProfit", "sum"),
            Orders=("Invoice", "nunique"),
            AveragePrice=("Price", "mean"),
        )
        .sort_values(["Period", "StockCode"])
    )
    return grouped

# Aggregate before any chronological train/test split.
daily_product = aggregate_products(product_sales, "D")
weekly_product = aggregate_products(product_sales, "W")

daily_product.to_csv(PROCESSED_DIR / "daily_product_features.csv", index=False)
weekly_product.to_csv(PROCESSED_DIR / "weekly_product_features.csv", index=False)

daily_product.shape, weekly_product.shape

((526114, 8), (196756, 8))

## Customer RFM Snapshots

Each row describes a customer using only sales on or before its `CutoffDate`. This prevents future purchases from leaking into customer features.

In [4]:
customer_sales = product_sales.loc[product_sales["Customer ID"].ne("Guest")].copy()
first_month = customer_sales["InvoiceDate"].min().to_period("M")
last_complete_month = customer_sales["InvoiceDate"].max().to_period("M") - 1
cutoffs = pd.date_range(
    first_month.to_timestamp("M"),
    last_complete_month.to_timestamp("M"),
    freq="ME",
)

rfm_snapshots = []
for cutoff in cutoffs:
    history = customer_sales.loc[customer_sales["InvoiceDate"] <= cutoff]
    snapshot = (
        history.groupby("Customer ID")
        .agg(
            LastPurchase=("InvoiceDate", "max"),
            Frequency=("Invoice", "nunique"),
            Monetary=("TotalPrice", "sum"),
        )
        .reset_index()
    )
    snapshot["CutoffDate"] = cutoff
    snapshot["Recency"] = (cutoff - snapshot["LastPurchase"]).dt.days
    rfm_snapshots.append(snapshot[["CutoffDate", "Customer ID", "Recency", "Frequency", "Monetary"]])

customer_rfm = pd.concat(rfm_snapshots, ignore_index=True)
customer_rfm.to_csv(PROCESSED_DIR / "customer_rfm_asof.csv", index=False)

assert (customer_rfm["Recency"] >= 0).all()
assert customer_rfm["CutoffDate"].max().month == 11
customer_rfm.shape

(91968, 5)